# Kubernetes

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-interview-questions/blob/master/notebooks/topics/kubernetes.ipynb)

*How to use this notebook.* Read the explanation cells and run the code cells below them. Python examples list their imports, so install what a section needs before running it. SQL and shell snippets are illustrative: they show the syntax for the platform named in that section and need that platform (or a compatible engine) to run.

Kubernetes is the default runtime for data platforms that need reproducible packaging, per-job isolation, and elastic compute. A data engineer rarely administers the whole cluster, but you are expected to read and write the manifests that run Spark drivers and executors, Airflow workers and triggerers, dbt jobs, Kafka/Flink operators, and the APIs around data products.

## Why is Kubernetes useful for data engineering workloads?

Kubernetes provides a consistent way to run containerized jobs, services, schedulers, and workers. Data teams use it for Airflow workers, Spark on Kubernetes, dbt jobs, custom ingestion services, and APIs around data products.

Example:

```text
CronJob -> runs daily Python ingestion container -> writes to object storage
```

The value is reproducible runtime packaging, resource limits, scaling, secrets management, and deployment consistency.

For data engineering specifically, four properties matter:

+ **Per-job resource isolation.** A Spark executor that needs 8 GiB and 4 CPU gets exactly that, and a run-away Python job cannot take the whole node.
+ **Elastic compute.** Jobs are scheduled on demand and nodes can be scaled up or down, so a cluster is not sized for peak batch load.
+ **One deployment model.** The same image and manifests move from dev to prod, and the same primitives (Jobs, CronJobs, Deployments) run batch and services.
+ **Ecosystem.** Spark, Flink, Airflow, dbt, Trino, Kafka, and most data tools already ship Helm charts or operators.

The trade-off is operational complexity: someone still owns upgrades, node pools, autoscaling, ingress, and the observability stack. A managed service (Databricks, EMR on EKS, Confluent Cloud) trades that complexity for cost and vendor coupling.

A cluster has two halves. The control plane runs the API server, the scheduler, and the controller-manager. Worker nodes run the kubelet, kube-proxy, and a container runtime. The API server is the only component anyone writes to; every other component watches it and reconciles toward the desired state you declared.

Follow one object to see the loop. You apply a Job and the API server stores it. The Job controller creates a Pod. The scheduler binds that Pod to a node with enough free requests. The kubelet on that node pulls the image, starts the container through the CRI, and reports status back. If the process exits non-zero, the kubelet applies the restart policy and the controller decides whether to retry. Nothing polls you.

```bash
kubectl apply -f orders-job.yaml
kubectl get job,pod -n data --watch
kubectl describe pod orders-etl-abc -n data
```

Common mistake: treating the cluster as one big machine. `kubectl get nodes` shows capacity per node, and the scheduler only places a pod where its requests fit on a single node. A job can sit `Pending` while the cluster looks half empty, because no individual node has the requested CPU or memory.

## What is the difference between a Kubernetes Job, CronJob, Deployment, and StatefulSet?

A Job runs a task to completion. A CronJob runs Jobs on a schedule. A Deployment manages stateless long-running services. A StatefulSet manages stateful services that need stable identities and persistent volumes.

Example choices:

+ Job: one-off backfill
+ CronJob: daily data extract
+ Deployment: metadata API service
+ StatefulSet: Kafka, Cassandra, or another clustered stateful service

A run-to-completion Job:

```yaml
apiVersion: batch/v1
kind: Job
metadata:
  name: backfill-orders-2026-09
  namespace: data
spec:
  backoffLimit: 3
  ttlSecondsAfterFinished: 86400
  template:
    spec:
      restartPolicy: Never
      containers:
        - name: backfill
          image: registry.example.com/data/orders-etl:1.14.2
          args: ["--start-date", "2026-09-01", "--end-date", "2026-09-30"]
          resources:
            requests:
              cpu: "2"
              memory: 4Gi
            limits:
              cpu: "4"
              memory: 8Gi
```

A scheduled Job:

```yaml
apiVersion: batch/v1
kind: CronJob
metadata:
  name: orders-daily-extract
  namespace: data
spec:
  schedule: "0 2 * * *"
  timeZone: "America/New_York"
  concurrencyPolicy: Forbid          # never overlap two loads of the same target
  successfulJobsHistoryLimit: 3
  failedJobsHistoryLimit: 3
  startingDeadlineSeconds: 600
  jobTemplate:
    spec:
      template:
        spec:
          restartPolicy: OnFailure
          containers:
            - name: extract
              image: registry.example.com/data/orders-etl:1.14.2
```

A Deployment manages stateless replicas; a StatefulSet adds stable identity, ordered rollout, and per-pod storage:

```yaml
apiVersion: apps/v1
kind: Deployment
metadata: {name: catalog-api, namespace: data}
spec:
  replicas: 3
  selector: {matchLabels: {app: catalog-api}}
  template:
    metadata: {labels: {app: catalog-api}}
    spec:
      containers:
        - name: api
          image: registry.example.com/data/catalog-api:2.3.0
          ports: [{containerPort: 8080}]
---
apiVersion: apps/v1
kind: StatefulSet
metadata: {name: kafka-broker, namespace: streaming}
spec:
  serviceName: kafka-headless
  replicas: 3
  selector: {matchLabels: {app: kafka}}
  template:
    metadata: {labels: {app: kafka}}
    spec:
      containers:
        - name: kafka
          image: registry.example.com/data/kafka:3.9
          volumeMounts: [{name: data, mountPath: /var/lib/kafka/data}]
  volumeClaimTemplates:
    - metadata: {name: data}
      spec:
        accessModes: ["ReadWriteOnce"]
        resources: {requests: {storage: 500Gi}}
```

For data pipelines, Jobs and CronJobs are common for batch tasks, while Deployments are common for always-on services. Use a StatefulSet only when the workload genuinely needs stable network identity or its own persistent volume; a job that reads and writes object storage is not stateful.

A Deployment never manages pods directly. It manages a ReplicaSet, and the ReplicaSet keeps the pod count at the desired value. Every change to the pod template creates a new ReplicaSet, and the Deployment moves replicas from the old one to the new one at the rate set by `strategy.rollingUpdate` (`maxSurge`, `maxUnavailable`). The old ReplicaSet is retained, which is why rollback works.

```bash
kubectl rollout status deploy/catalog-api -n data
kubectl rollout history deploy/catalog-api -n data
kubectl set image deploy/catalog-api api=registry.example.com/data/catalog-api:2.4.0 -n data
kubectl rollout undo deploy/catalog-api -n data
```

A DaemonSet runs one pod on every eligible node and is how node agents ship: log collectors, metrics exporters, CSI drivers, CNI plugins. Control-plane taints keep it off the control plane unless the template tolerates them.

A normal pod moves through phases: `Pending`, `Running`, `Succeeded`, `Failed`. `restartPolicy` is set per pod and applies per container: `Always` (the default for Deployments), `OnFailure`, and `Never`. A Job pod template must use `OnFailure` or `Never`, because `Always` restarts a finished task forever. Under `Always` the container restarts inside the same pod, so the pod name does not change.

Common mistake: reading a rising RESTARTS count as a new pod. A Deployment keeps the same pod and restarts the container; read the last terminated state to learn why.

```bash
kubectl get pod orders-etl-abc -n data -o wide
kubectl get pod orders-etl-abc -n data -o jsonpath='{.status.containerStatuses[*].restartCount}'
kubectl describe pod orders-etl-abc -n data   # Last State: Terminated, Reason: OOMKilled
```

## How to do maintenance activity on the K8 node?

Whenever there are security patches available the Kubernetes administrator has to perform the maintenance task to apply the security patch to the running container in order to prevent it from vulnerability, which is often an unavoidable part of the administration. The following two commands are useful to safely drain the K8s node.

```bash
kubectl cordon <node-name>
kubectl drain <node-name> --ignore-daemonsets --delete-emptydir-data
```

The first command moves the node to maintenance mode or makes the node unavailable, followed by `kubectl drain` which will finally discard the pod from the node. After the drain command is a success you can perform maintenance.

Note: If you wish to perform maintenance on a single pod following two commands can be issued in order:

```bash
kubectl get nodes                  # list all the nodes
kubectl drain <node-name>          # drain a particular node
kubectl uncordon <node-name>       # return it to service afterwards
```

Operational notes that matter for data workloads:

+ `--ignore-daemonsets` is required because DaemonSet pods cannot be moved.
+ Draining respects PodDisruptionBudgets, so a Kafka cluster with `minAvailable: 2` will block eviction if only two brokers are healthy.
+ A Spark executor pod is usually evicted cleanly; the driver retries the task. A long-running stateful pod needs its operator or a manual leader step-down first.
+ On managed platforms, node auto-repair and maintenance windows do this for you; use `kubectl get events` and the node conditions (`MemoryPressure`, `DiskPressure`) to confirm.

## How do persistent volumes and storage classes work?

A container filesystem dies with the container, so stateful data needs a volume. A PersistentVolumeClaim (PVC) is a namespaced request for storage. A PersistentVolume (PV) is the backing volume, created ahead of time by an administrator or provisioned on demand by a CSI driver through a StorageClass. The PVC binds to one PV and keeps it until the claim is deleted. Access modes are `ReadWriteOnce` (one node), `ReadOnlyMany`, `ReadWriteMany` (shared filesystem), and the newer `ReadWriteOncePod`.

A StorageClass decides when provisioning happens. `volumeBindingMode: Immediate` creates the volume as soon as the PVC exists, which can place it in a zone the pod later cannot reach. `WaitForFirstConsumer` waits for the pod and provisions in that pod's zone, which is the safer default in a multi-zone cluster.

```yaml
apiVersion: v1
kind: PersistentVolumeClaim
metadata:
  name: kafka-data-0
  namespace: streaming
spec:
  accessModes: ["ReadWriteOnce"]
  storageClassName: fast-ssd
  resources:
    requests:
      storage: 500Gi
---
apiVersion: v1
kind: Pod
metadata:
  name: kafka-broker-0
spec:
  containers:
    - name: kafka
      image: registry.example.com/data/kafka:3.9
      volumeMounts:
        - name: data
          mountPath: /var/lib/kafka/data
  volumes:
    - name: data
      persistentVolumeClaim:
        claimName: kafka-data-0
```

In a StatefulSet, `volumeClaimTemplates` creates one PVC per replica, named `<template>-<statefulset>-<ordinal>`, and deleting the StatefulSet does not delete those PVCs. That is deliberate: the data outlives the pod.

Common mistake: expecting a `ReadWriteOnce` claim to mount on two pods on different nodes. RWO is per node, not per pod, so the second pod stays `ContainerCreating` or reports a multi-attach error. Use `ReadWriteMany` on a shared filesystem, or give each replica its own claim. Check `allowVolumeExpansion` on the class before you rely on resizing a full volume.

## How do we control the resource usage of POD?

With the use of limit and request resource usage of a POD can be controlled.

Request: The number of resources being requested for a container. If a container exceeds its request for resources, it can be throttled back down to its request.

Limit: An upper cap on the resources a single container can use. If it tries to exceed this predefined limit it can be terminated if K8's decides that another container needs these resources. If you are sensitive towards pod restarts, it makes sense to have the sum of all container resource limits equal to or less than the total resource capacity for your cluster.

Example:

```yaml
apiVersion: v1
kind: Pod
metadata:
  name: demo
spec:
  containers:
    - name: example1
      image: example/example1
      resources:
        requests:
          memory: "512Mi"
          cpu: "250m"
        limits:
          memory: "1Gi"
          cpu: "500m"
```

How the scheduler and the kernel actually use these values:

+ **requests** decide scheduling and are the minimum guaranteed share. CPU requests set `cpu.shares`; memory requests influence eviction ordering.
+ **limits** set the hard ceiling. A container that exceeds a memory limit is OOM-killed; a container that exceeds a CPU limit is throttled, not killed.
+ **QoS class** follows from the combination:
  + `Guaranteed` - requests equal limits on every container. Last to be evicted.
  + `Burstable` - requests set but below limits (the common case).
  + `BestEffort` - nothing set. First to be evicted, and a bad choice for production data jobs.
+ **Namespaces and quotas** cap an entire team, not one pod:

```yaml
apiVersion: v1
kind: ResourceQuota
metadata:
  name: data-team-quota
  namespace: data
spec:
  hard:
    requests.cpu: "200"
    requests.memory: 800Gi
    limits.cpu: "400"
    limits.memory: 1600Gi
    count/jobs.batch: "50"
```

A realistic Spark executor block sizes both values so the JVM heap, off-heap memory, and container limit line up:

```yaml
resources:
  requests:
    cpu: "4"
    memory: "16Gi"
  limits:
    cpu: "8"
    memory: "18Gi"
```

Interview point: memory requests and limits must account for JVM overhead, not just `spark.executor.memory`. If the container limit is below the JVM's total footprint, the pod restarts with exit code 137 (OOMKilled) even though Spark shows no OOM.

`cordon` marks a node unschedulable but leaves its pods running. `drain` evicts the pods and then cordons the node. Eviction is a call to the pod's `/eviction` subresource, so it respects PodDisruptionBudgets and waits for `terminationGracePeriodSeconds`; a pod whose budget cannot be met is never evicted and the drain blocks.

Worked example: see what will move before you patch a node.

```bash
kubectl get pods -n data -o wide --field-selector spec.nodeName=ip-10-0-3-17
kubectl cordon ip-10-0-3-17
kubectl drain ip-10-0-3-17 --ignore-daemonsets --delete-emptydir-data --dry-run=client
kubectl drain ip-10-0-3-17 --ignore-daemonsets --delete-emptydir-data --timeout=10m
kubectl uncordon ip-10-0-3-17
```

Common mistake: draining several nodes at once. Evicted pods need somewhere to land; if too much of the cluster is cordoned they stay `Pending` and the pipeline misses its window. Drain one node at a time, watch `kubectl get pods -A -o wide`, and remember that `--delete-emptydir-data` throws away scratch shuffle data.

## What are the various K8's services running on nodes and describe the role of each service?

Mainly K8 cluster consists of two types of nodes, executor and master.

Executor node:

Kube-proxy: This service is responsible for the communication of pods within the cluster and to the outside network, which runs on every node. This service is responsible to maintain network protocols when your pod establishes a network communication.

kubelet: Each node has a running kubelet service that updates the running node accordingly with the configuration(YAML or JSON) file. NOTE: kubelet service is only for containers created by Kubernetes.

Master services:

Kube-apiserver: Master API service which acts as an entry point to K8 cluster.

Kube-scheduler: Schedule PODs according to available resources on executor nodes.

Kube-controller-manager: is a control loop that watches the shared state of the cluster through the apiserver and makes changes attempting to move the current state towards the desired stable state.

The container runtime (containerd, CRI-O) is the piece that actually pulls images and runs containers; kubelet talks to it through the CRI interface. On managed clouds the master nodes are hidden behind an SLA, but the same components exist.

```bash
kubectl get pods -n kube-system
kubectl get componentstatuses        # apiserver, scheduler, controller-manager health
kubectl top nodes
kubectl top pods -n data
```

For a data engineer, the actionable version is: if pods stay `Pending`, check the scheduler; if they stay `ContainerCreating`, check kubelet events and image pulls; if the API server is down, nothing in your pipeline can be reconciled.

The two numbers are used by different systems. The scheduler sums `requests` to pick a node, so requests are the packing unit. The kubelet writes both into the container cgroup: a CPU request becomes `cpu.weight` or `cpu.shares`, a CPU limit becomes the CFS quota (`cpu.max`, a budget per 100 ms period), and a memory limit becomes `memory.max`. A container that burns its quota inside a period is throttled until the next period even if the node is idle.

Memory has no equivalent of throttling. When a container working set exceeds `memory.max`, the kernel OOM killer ends a process in that cgroup, the kubelet reports `OOMKilled`, and the container exits 137. When a whole node runs short, the kubelet evicts pods in QoS order: BestEffort first, then Burstable ordered by how far usage exceeds the request, and Guaranteed last. A memory request is a protection number, not just a scheduling number.

Common mistake: setting a CPU limit barely above the request for a multi-threaded job. A Spark executor with a 4-core limit and many threads hits the 100 ms quota, gets throttled, and slows shuffle and JVM work while the node still has spare CPU. Leave headroom, or set a request and no CPU limit, and watch throttling instead of guessing.

kubelet is the only component that creates containers. It watches the API server for pods bound to its node, calls the container runtime through the CRI, mounts volumes, runs probes, and posts status back. The scheduler never talks to a node; it only writes a binding to the API server. kube-proxy watches Services and EndpointSlices and programs packet rules so a virtual IP reaches one healthy pod.

Worked example: find which layer is failing.

```bash
kubectl get --raw='/readyz?verbose'          # API server gates
kubectl get pods -n kube-system -o wide      # control plane and CNI
kubectl get pod orders-etl-abc -n data -o wide
kubectl describe node ip-10-0-3-17           # Conditions: MemoryPressure, DiskPressure
```

Common mistake: blaming the application when a pod never starts. `Pending` means the scheduler could not place it. `ContainerCreating` means the kubelet is stuck on an image pull, a volume mount, or the CNI. `Running` with failing probes is the container's own problem. Check `kubectl get events` before reading application logs.

## What is PDB (Pod Disruption Budget)?

A Kubernetes administrator can create a deployment of a kind: PodDisruptionBudget for high availability of the application, it makes sure that the minimum number is running pods are respected as mentioned by the attribute minAvailable spec file. This is useful while performing a drain where the drain will halt until the PDB is respected to ensure the High Availability(HA) of the application. The following spec file also shows minAvailable as 2 which implies the minimum number of an available pod (even after the election).

Example: YAML Config using minAvailable:

```yaml
apiVersion: policy/v1
kind: PodDisruptionBudget
metadata:
  name: zk-pdb
  namespace: streaming
spec:
  minAvailable: 2
  selector:
    matchLabels:
      app: zookeeper
```

The alternative is `maxUnavailable`:

```yaml
apiVersion: policy/v1
kind: PodDisruptionBudget
metadata:
  name: trino-worker-pdb
  namespace: analytics
spec:
  maxUnavailable: 1
  selector:
    matchLabels:
      app: trino-worker
```

Notes:

+ A PDB only protects against *voluntary* disruptions (drain, node upgrade). It cannot stop a node crash or an OOM kill.
+ `minAvailable` as a percentage (`80%`) scales better than a fixed number as replicas change.
+ Pair a PDB with anti-affinity so the replicas are actually on different nodes; otherwise the budget can never be satisfied.
+ PDBs are how a broker or a query worker survives a rolling node upgrade without a full outage.

The budget is the promise, the selector is the scope, and the deployment is the reality. With three Kafka brokers, `minAvailable: 2` lets a drain evict one broker at a time. `maxUnavailable: 1` expresses the same thing from the other side and adapts better when replicas grow. A percentage scales with the workload; a fixed number does not.

The budget only constrains voluntary disruptions: `kubectl drain`, node upgrades, and the eviction API. A node crash, a spot reclaim, or an OOM kill ignores it. A PDB whose selector matches nothing protects nothing, and a PDB on a single-replica Deployment with `minAvailable: 1` blocks every drain forever.

Common mistake: writing a PDB without anti-affinity. If all three brokers run on one node, `minAvailable: 2` can never be satisfied during that node's drain. Pair the budget with pod anti-affinity or a topology spread constraint so the replicas are genuinely separate.

## What's the init container and when it can be used?

init containers will set a stage for you before running the actual POD: they run to completion, in order, before any app container starts, and each must succeed before the next begins.

Common uses:

+ Wait for a dependency (a database, a schema version, a lock) before the app starts.
+ Clone a git repository or download a model artifact into a shared volume.
+ Fix ownership/permissions on a mounted volume (run as root, then drop to a non-root user).

```yaml
apiVersion: v1
kind: Pod
metadata:
  name: dbt-runner
spec:
  initContainers:
    - name: fetch-dbt-project
      image: registry.example.com/data/git-sync:1.2
      command: ["git", "clone", "--depth", "1",
                "https://github.com/acme/analytics-dbt.git", "/workspace"]
      volumeMounts:
        - name: workspace
          mountPath: /workspace
  containers:
    - name: dbt
      image: registry.example.com/data/dbt:1.9
      command: ["dbt", "run", "--project-dir", "/workspace"]
      volumeMounts:
        - name: workspace
          mountPath: /workspace
  volumes:
    - name: workspace
      emptyDir: {}
```

When *not* to use one: init containers add startup time and are per-pod, not per-container. For a fast health check on a long-running service, use a readiness probe instead, because an init container that blocks delays the whole pod rather than removing it from load balancing.

Init containers run to completion, one after another, before the app containers start. Each must succeed or the pod is retried according to `restartPolicy`. They share the pod network and can mount the same volumes, including `emptyDir` scratch space, which is how an artifact is fetched or permissions are fixed before the real process starts.

Sidecars are the other half of the story. A plain second container runs alongside the app for the pod's whole life. Since Kubernetes 1.29 an entry in `initContainers` may set `restartPolicy: Always`, which makes it a native sidecar: it starts before the app and keeps running after the app containers exit. That is the supported way to run a proxy or a log shipper next to a job.

Common mistake: putting a long-running process in an ordinary init container. It must exit before the pod can start, so a fetch that never returns leaves the pod stuck in `Init:0/1`. Use a native sidecar when the process should stay up.

## What is the role of Load Balance in Kubernetes?

Load balancing is a way to distribute the incoming traffic into multiple backend servers, which is useful to ensure the application available to the users.

In Kubernetes, all the incoming traffic lands to a single IP address on the load balancer which is a way to expose your service to outside the internet which routes the incoming traffic to a particular pod (via service) using an algorithm known as round-robin. Even if any pod goes down load balances are notified so that the traffic is not routed to that particular unavailable node. Thus load balancers in Kubernetes are responsible for distributing a set of tasks (incoming traffic) to the pods.

Three load-balancing layers exist, and interviewers like candidates who separate them:

+ **kube-proxy / Service (L4).** A `ClusterIP` service is a virtual IP; kube-proxy programs iptables/IPVS rules so each connection is forwarded to one healthy endpoint selected by round-robin (or session affinity if configured).
+ **Cloud load balancer (L4).** A `Service` of `type: LoadBalancer` asks the cloud provider to provision an external LB that targets the nodes or the pod IPs.
+ **Ingress / Gateway (L7).** An ingress controller terminates HTTP(S), routes by host/path, and can do TLS, retries, and header-based rules.

```yaml
apiVersion: v1
kind: Service
metadata:
  name: warehouse-api
  namespace: data
spec:
  type: ClusterIP
  selector:
    app: warehouse-api
  ports:
    - name: http
      port: 80
      targetPort: 8080
```

Readiness probes are what make load balancing correct: a pod that is not `Ready` is removed from the endpoint list, so traffic only reaches pods that can serve it.

A Service is backed by EndpointSlices, which kube-proxy turns into forwarding rules. A pod is added only when its readiness probe passes and its labels match the Service selector. `ClusterIP` is the default and is reachable only inside the cluster. `sessionAffinity: ClientIP` pins a client to one pod, which matters for stateful protocols.

Worked example: an empty endpoint list is the usual cause of a 503.

```bash
kubectl get svc,endpointslices -n data
kubectl get endpointslices -n data -l kubernetes.io/service-name=warehouse-api
kubectl get pods -n data -l app=warehouse-api --show-labels
kubectl describe pod -n data -l app=warehouse-api
```

Common mistake: a selector typo. `spec.selector` on the Service and `metadata.labels` on the pod must match exactly. When they do not, the Service has no endpoints, traffic is dropped, and the pods look perfectly healthy.

## What are the various things that can be done to increase Kubernetes security?

By default, POD can communicate with any other POD, we can set up network policies to limit this communication between the PODs.

RBAC (Role-based access control) to narrow down the permissions.

Use namespaces to establish security boundaries.

Set the admission control policies to avoid running the privileged containers.

Turn on audit logging.

Concrete controls a data platform should actually enable:

```yaml
apiVersion: networking.k8s.io/v1
kind: NetworkPolicy
metadata:
  name: allow-data-to-warehouse-only
  namespace: data
spec:
  podSelector:
    matchLabels: {app: orders-etl}
  policyTypes: [Egress]
  egress:
    - to:
        - namespaceSelector:
            matchLabels: {kubernetes.io/metadata.name: warehouse}
      ports:
        - port: 5432
    - to:
        - ipBlock:
            cidr: 10.0.0.0/8
      ports:
        - port: 443
```

```yaml
securityContext:
  runAsNonRoot: true
  runAsUser: 10001
  readOnlyRootFilesystem: true
  allowPrivilegeEscalation: false
  capabilities:
    drop: ["ALL"]
  seccompProfile:
    type: RuntimeDefault
```

Also worth stating: least-privilege service accounts with short-lived credentials (IRSA, Workload Identity) instead of long-lived cloud keys in Secrets; image scanning and a signed-image policy; `ResourceQuota` and `LimitRange` to stop crypto-miner-style abuse; and namespace-level RBAC so a notebook user cannot read production Secrets. The most common real incident is not a cluster break-in - it is a job with a wildcard cloud role and a plaintext credential in a ConfigMap.

RBAC answers who may call which API, Pod Security Admission answers what a pod may look like, and NetworkPolicy answers who may talk to whom. The `restricted` Pod Security Standard enforces non-root, dropped capabilities, a read-only root filesystem, and a seccomp profile; `baseline` blocks the obvious escapes; `privileged` is unrestricted. Apply the level per namespace with labels such as `pod-security.kubernetes.io/enforce: restricted`.

The data-platform specifics are credentials and blast radius. Prefer projected service account tokens over long-lived Secrets, give each workload its own service account, and deny pod-creation rights outside the operator that needs them. An Airflow worker that can create pods in any namespace can mount any secret in the cluster.

Common mistake: a wildcard ClusterRole for a controller. `create pods` looks narrow, but combined with `get secrets` it becomes cluster-wide credential access. Bind Roles instead of ClusterRoles, and name the resources.

## How to monitor the Kubernetes cluster?

Prometheus is used for Kubernetes monitoring. The Prometheus ecosystem consists of multiple components.

Mainly Prometheus server which scrapes and stores time-series data.

Client libraries for instrumenting application code.

Push gateway for supporting short-lived jobs.

Special-purpose exporters for services like StatsD, HAProxy, Graphite, etc.

An alert manager to handle alerts on various support tools.

The modern stack on top of that is:

+ **kube-state-metrics** for object state: desired vs available replicas, Job failures, Pending pods, PVC status, CronJob last-successful time.
+ **node-exporter** for node CPU, memory, disk, and network.
+ **cAdvisor / metrics-server** for per-container usage, which powers `kubectl top` and the HPA.
+ **Grafana** for dashboards, **Alertmanager** for routing.

Signals a data platform should alert on:

| Signal | Why it matters |
|---|---|
| Job `Failed` / `backoffLimitExceeded` | a pipeline run failed silently |
| CronJob `lastSuccessfulTime` too old | the schedule stopped firing |
| Pod `OOMKilled` (exit 137) | executor memory is under-provisioned |
| `Pending` pods for more than N minutes | unschedulable: capacity or taints |
| Node `MemoryPressure` / `DiskPressure` | an imminent eviction storm |
| HPA at max replicas | capacity ceiling reached |

```text
# pods OOM-killed in the last hour, by workload
sum by (namespace, pod) (
  kube_pod_container_status_last_terminated_reason{reason="OOMKilled"}
)
```

Three sources cover different questions. cAdvisor, exposed through the kubelet, gives per-container CPU, memory, and throttling; the metrics-server aggregates it for `kubectl top` and the HPA. node-exporter gives node CPU, memory, disk, and network. kube-state-metrics turns API objects into time series: desired versus available replicas, Job failures, Pending pods, PVC phase, and CronJob last-successful time. Prometheus scrapes all three and Alertmanager routes the alerts.

In [ ]:
# ILLUSTRATIVE - requires kubernetes: install to run, or read on.
# Review only: the kubernetes client is not installed in this repository.
# This shows the API shape used to read pod and container state from Python.
from kubernetes import client, config

config.load_kube_config()
core = client.CoreV1Api()
pods = core.list_namespaced_pod("data", label_selector="app=spark-executor")
for pod in pods.items:
    for status in pod.status.container_statuses or []:
        terminated = status.last_state.terminated
        print(pod.metadata.name, status.name, status.restart_count,
              terminated.reason if terminated else None)

Common mistake: alerting on CPU utilization alone. A throttled container can show modest CPU use while latency climbs, because the CFS quota stopped it. Watch `container_cpu_cfs_throttled_seconds_total` divided by `container_cpu_cfs_periods_total`, and alert on restart count and `OOMKilled` rather than on a dashboard nobody reads.

## How to get the central logs from POD?

This architecture depends upon the application and many other factors. Following are the common logging patterns:

Node level logging agent.

Streaming sidecar container.

Sidecar container with the logging agent.

Export logs directly from the application.

In the setup, journalbeat and filebeat are running as daemonset. Logs collected by these are dumped to the kafka topic which is eventually dumped to the ELK stack. The same can be achieved using EFK stack and fluentd-bit.

```yaml
apiVersion: apps/v1
kind: DaemonSet
metadata:
  name: fluent-bit
  namespace: logging
spec:
  selector:
    matchLabels: {app: fluent-bit}
  template:
    metadata:
      labels: {app: fluent-bit}
    spec:
      tolerations:
        - operator: Exists          # run on every node, including tainted ones
      containers:
        - name: fluent-bit
          image: cr.fluentbit.io/fluent/fluent-bit:3.1
          volumeMounts:
            - name: varlog
              mountPath: /var/log
            - name: containers
              mountPath: /var/lib/docker/containers
              readOnly: true
      volumes:
        - name: varlog
          hostPath: {path: /var/log}
        - name: containers
          hostPath: {path: /var/lib/docker/containers}
```

Practical guidance for data jobs:

+ Prefer stdout/stderr logging from the container; the node agent collects it without any app change.
+ Add a correlation key (`run_id`, `dag_id`, `task_id`, `spark.app.id`) to every line so a distributed job is searchable across driver and executors.
+ Kubernetes log retention on the node is short. Ship logs to durable storage (object storage, ClickHouse, Elasticsearch, Loki) before they roll.
+ For Spark on Kubernetes, enable the log aggregation (or the Spark history server) or the executor logs disappear with the pod.

```bash
kubectl logs deploy/orders-etl -n data --since=1h
kubectl logs -l spark-role=executor -n data --tail=200
```

A container's stdout and stderr go to a file on the node under `/var/log/pods`, and the runtime rotates it. `kubectl logs` streams from the kubelet, so it sees only the current container unless you ask for `--previous`. Logs disappear when the pod is deleted, which is why a node-level DaemonSet ships them to durable storage.

```bash
kubectl logs deploy/orders-etl -n data --since=30m
kubectl logs orders-etl-abc -n data -c extract          # one container of several
kubectl logs orders-etl-abc -n data --previous          # the crashed instance
kubectl logs -l app=spark-executor -n data --tail=200 --prefix
kubectl logs orders-etl-abc -n data --timestamps | head
```

Common mistake: deleting a `CrashLoopBackOff` pod and losing the evidence with it. The current container may have barely started; `--previous` holds the output from the instance that failed, including the stack trace or the config error.

## How to turn the service defined below in the spec into an external one?

Starting spec:

```yaml
spec:
  selector:
    app: some-app
  ports:
    - protocol: UDP
      port: 8080
      targetPort: 8080
```

Explanation: with no `type`, this is a `ClusterIP` service and is only reachable inside the cluster. Adding `type: LoadBalancer` (with an optional `nodePort`) makes it external:

```yaml
apiVersion: v1
kind: Service
metadata:
  name: some-app
spec:
  selector:
    app: some-app
  type: LoadBalancer
  ports:
    - protocol: UDP
      port: 8080
      targetPort: 8080
      nodePort: 32412        # optional; omit to let Kubernetes allocate one
```

Complete the following configuration spec file to make it an Ingress:

```yaml
apiVersion: networking.k8s.io/v1
kind: Ingress
metadata:
  name: someapp-ingress
  annotations:
    nginx.ingress.kubernetes.io/rewrite-target: /
spec:
  ingressClassName: nginx
  rules:
    - host: my.host
      http:
        paths:
          - path: /
            pathType: Prefix
            backend:
              service:
                name: some-app
                port:
                  number: 8080
```

One of the several ways to answer this question. Choose by protocol and layer: `NodePort` for quick tests (one port on every node), `LoadBalancer` for a raw L4 endpoint, `Ingress` for HTTP routing, TLS, and host/path rules. The old `serviceName`/`servicePort` fields shown in older docs were replaced by the nested `service.name`/`service.port.number` shape in `networking.k8s.io/v1`.

Choose by protocol and exposure. `NodePort` opens one port on every node and is fine for a quick test. `LoadBalancer` asks the cloud controller for an L4 endpoint and costs money per service. `Ingress` is L7 and routes by host and path through a controller, so many services share one load balancer. `externalTrafficPolicy: Local` preserves the client IP but routes only to pods on the receiving node.

Worked example: the same app moving outward.

```bash
kubectl patch svc some-app -p '{"spec":{"type":"LoadBalancer"}}'
kubectl get svc some-app --watch          # wait for EXTERNAL-IP
kubectl get ingress -n data
kubectl describe ingress someapp-ingress -n data
```

Common mistake: exposing a data service with `type: LoadBalancer` and no auth. Every L4 service gets a public address by default. Prefer Ingress with TLS and an auth layer, or keep the service internal and reach it through a bastion or `kubectl port-forward`.

## How to configure TLS with Ingress?

Add `tls` and `secretName` entries. The secret must be a `kubernetes.io/tls` secret containing `tls.crt` and `tls.key`.

```yaml
apiVersion: networking.k8s.io/v1
kind: Ingress
metadata:
  name: someapp-ingress
spec:
  ingressClassName: nginx
  tls:
    - hosts:
        - some_app.com
      secretName: someapp-secret-tls
  rules:
    - host: some_app.com
      http:
        paths:
          - path: /
            pathType: Prefix
            backend:
              service:
                name: some-app
                port:
                  number: 8080
```

```bash
# create the TLS secret (or let cert-manager do it)
kubectl create secret tls someapp-secret-tls \
  --cert=tls.crt --key=tls.key -n data

# redirect HTTP to HTTPS
# annotation: nginx.ingress.kubernetes.io/ssl-redirect: "true"
```

For production, do not manage certificates by hand: install cert-manager and issue `Certificate` resources from Let's Encrypt or your internal CA, and let it rotate the secret. Note that TLS terminates at the ingress; if you also want encryption inside the cluster, configure the backend protocol (`nginx.ingress.kubernetes.io/backend-protocol: "HTTPS"`) or a service mesh.

TLS terminates at the ingress controller. The controller reads `spec.tls[].secretName` from the same namespace as the Ingress and expects a `kubernetes.io/tls` secret with `tls.crt` and `tls.key`. The `hosts` list must cover every host in `rules`, or the controller serves its default certificate.

cert-manager turns this into a Certificate resource and renews it before expiry, writing the secret itself so nothing is rotated by hand.

```yaml
apiVersion: cert-manager.io/v1
kind: Certificate
metadata:
  name: someapp-tls
  namespace: data
spec:
  secretName: someapp-secret-tls
  dnsNames: ["some_app.com"]
  issuerRef:
    name: letsencrypt-prod
    kind: ClusterIssuer
```

Common mistake: creating the TLS secret in the `default` namespace while the Ingress lives in `data`. The controller cannot read across namespaces, so it silently serves a self-signed default and the browser warning is misread as an application bug.

## Why use namespaces? What is the problem with using the default namespace?

While using the default namespace alone, it becomes hard over time to get an overview of all the applications you can manage in your cluster. Namespaces make it easier to organize the applications into groups that make sense, like a namespace of all the monitoring applications and a namespace for all the security applications, etc.

Namespaces can also be useful for managing Blue/Green environments where each namespace can include a different version of an app and also share resources that are in other namespaces (namespaces like logging, monitoring, etc.).

Another use case for namespaces is one cluster with multiple teams. When multiple teams use the same cluster, they might end up stepping on each other's toes. For example, if they end up creating an app with the same name it means one of the teams overrides the app of the other team because there can't be two apps in Kubernetes with the same name (in the same namespace).

Namespaces are also the unit of the controls a data platform cares about most:

+ `ResourceQuota` and `LimitRange` per team.
+ RBAC `Role`/`RoleBinding` scoped to a team.
+ NetworkPolicy isolation between tenants.
+ A `default` namespace gets none of this and is shared by every chart that forgets to set `namespace:`, which is exactly why workloads there are painful to attribute, budget, or delete safely.

A namespace scopes names, RBAC, quotas, and NetworkPolicy selection, and it is part of cluster DNS (`service.namespace.svc.cluster.local`). It is not a hard security boundary on its own: pods in different namespaces can reach each other until a NetworkPolicy says otherwise, and a ClusterRoleBinding ignores namespace lines entirely.

Worked example: a team namespace with guardrails.

```bash
kubectl create namespace data-payments
kubectl label namespace data-payments pod-security.kubernetes.io/enforce=restricted
kubectl create quota payments-quota -n data-payments \
  --hard=requests.cpu=40,requests.memory=160Gi,limits.cpu=80,limits.memory=320Gi,pods=120
kubectl create rolebinding payments-edit -n data-payments \
  --clusterrole=edit --group=payments-engineers
```

Common mistake: leaving everything in `default`. Names collide with another team, quotas cannot be attributed, and charts that forget a namespace land in the same pile. Give every team and every environment its own namespace.

## In the following file which service and in which namespace is referred?

```yaml
apiVersion: v1
kind: ConfigMap
metadata:
  name: some-configmap
data:
  some_url: silicon.chip
```

Answer - It's referencing the service "silicon" in the namespace called "chip". The `name.namespace` form is the DNS shorthand used inside a cluster; a fully qualified name would be `silicon.chip.svc.cluster.local`. A bare `silicon` would mean the service in the *same* namespace as the pod doing the lookup.

The pod resolver appends search domains from `/etc/resolv.conf`, so the number of dots changes what a name means. `silicon` is a service in the pod's own namespace. `silicon.chip` is a service in the `chip` namespace. `silicon.chip.svc.cluster.local` is fully qualified and always unambiguous. A headless Service (`clusterIP: None`) returns pod IPs directly so clients can balance themselves.

Common mistake: using a bare service name from a pod in another namespace. It resolves to nothing and looks like a network failure. Use the `name.namespace` form, or set `spec.dnsConfig` and shorten the name deliberately.

## What is an Operator?

"Operators are software extensions to K8s which make use of custom resources to manage applications and their components. Operators follow Kubernetes principles, notably the control loop."

In practice an Operator = Custom Resource Definition (CRD) + a controller that reconciles the desired state. Data-platform examples: Strimzi (Kafka), Flink Kubernetes Operator, Spark Operator, CloudNativePG, Airflow's official Helm chart plus community operators.

```yaml
apiVersion: kafka.strimzi.io/v1beta2
kind: Kafka
metadata:
  name: events
  namespace: streaming
spec:
  kafka:
    replicas: 3
    storage:
      type: jbod
      volumes:
        - id: 0
          type: persistent-claim
          size: 1Ti
          deleteClaim: false
    config:
      min.insync.replicas: 2
  zookeeper:
    replicas: 3
```

Applying that one object creates StatefulSets, Services, ConfigMaps, and PDBs, and the controller keeps them converged - including safe rolling restarts and rebalancing.

An operator is a controller plus one or more Custom Resource Definitions. You declare the desired state in a custom object; the controller watches it, creates and updates the built-in objects that realize it, and writes progress back into the object `status`. A finalizer lets it clean up external resources before the object is deleted. Reconciliation is level-based, so the controller survives restarts and drift.

```bash
kubectl get crds | grep -E 'spark|flink|strimzi'
kubectl get sparkapplications -n data
kubectl describe flinkdeployment vehicle-enrichment -n streaming
kubectl get pods -n operators
```

Common mistake: treating a CRD as a normal manifest. CRDs are cluster-scoped and versioned; deleting and recreating one can delete every custom object of that kind. Upgrade the operator and its CRDs together, and read the conversion notes.

## Why do we need Operators?

The process of managing applications in Kubernetes isn't as straightforward as managing stateless applications, where reaching the desired status and upgrades are both handled the same way for every replica. In stateful applications, upgrading each replica might require different handling due to the stateful nature of the app, each replica might be in a different status. As a result, we often need a human operator to manage stateful applications. Kubernetes Operator is supposed to assist with this.

This will also help with automating a standard process on multiple Kubernetes clusters.

What an operator buys you for a data system:

+ **Domain-aware rolling upgrades.** Strimzi drains a Kafka broker and waits for under-replicated partitions before moving on; a plain Deployment would restart all brokers at once.
+ **Ordering and quorum.** ZooKeeper/KRaft, Cassandra, and Elasticsearch need a specific start order and cluster-formation logic.
+ **Day-2 tasks.** Scaling a cluster, rebalancing, taking a snapshot, and triggering a backup become CRD fields instead of runbooks.
+ **Drift correction.** The control loop re-creates a deleted broker or re-attaches a lost volume.

The cost is that an operator is code you did not write; pin its version, review its CRDs on upgrade, and understand its failure modes before you trust it with a tier-1 cluster.

Use an operator when the application has domain knowledge a generic controller lacks: a broker that must drain partitions before restarting, a database that must form quorum in order, a stream processor that must take a savepoint before changing parallelism. For an ordinary stateless service a Deployment is simpler and better understood.

Common mistake: adopting an operator and then hand-editing its child StatefulSets. The next reconcile overwrites the change. Drive everything through the custom resource, or the operator becomes a source of surprise.

## What is GKE?

GKE is Google Kubernetes Engine that is used for managing and orchestrating systems for Docker containers. With the help of Google Public Cloud, we can also orchestrate the container cluster. It is a managed control plane: Google runs the API server, scheduler, and controller-manager, and you pay for the worker node pools (or use Autopilot, where Google also manages the nodes).

Data-platform equivalents: **EKS** on AWS and **AKS** on Azure. The relevant managed features for data workloads are node auto-provisioning, spot/preemptible node pools, workload identity for passwordless access to object storage and warehouses, and cluster autoscaling that lets a Spark backfill burst and then release the nodes.

A managed control plane means the API server, scheduler, and controller-manager are someone else's uptime problem. What remains yours is the node layer: node pools with instance types and taints, autoscaling settings, and the identity wiring that lets a pod read object storage without a static key. GKE Autopilot removes the node layer too and bills per pod request, which changes how you size workloads.

Common mistake: assuming managed means no capacity planning. A managed cluster still has a fixed node pool by default; Spark executors stay `Pending` until node auto-provisioning or the cluster autoscaler adds nodes, and that takes minutes.

## What is Ingress Default Backend?

It specifies what to do with an incoming request to the Kubernetes cluster that isn't mapped to any backend i.e what to do when no rules being defined for the incoming HTTP request. If the default backend service is not defined, it's recommended to define it so that users still see some kind of message instead of an unclear error.

```yaml
apiVersion: networking.k8s.io/v1
kind: Ingress
metadata:
  name: catch-all
spec:
  ingressClassName: nginx
  defaultBackend:
    service:
      name: maintenance-page
      port:
        number: 80
```

A default backend returns a controlled 404 or a maintenance page for unknown hosts/paths. Without one, users see the ingress controller's raw default (nginx 404), which is confusing and can leak controller details.

A default backend answers requests that match no rule, so unknown hosts and stray paths get a controlled response instead of the controller raw 404. It is a normal Service reference and can point at a maintenance page. For nginx it can also be set with the `nginx.ingress.kubernetes.io/default-backend` annotation.

Common mistake: relying on a catch-all path rule and forgetting the default backend. A request with an unexpected `Host` header then reaches whatever the controller does by default, which is confusing at best and can leak version banners.

## How to run Kubernetes locally?

Kubernetes can be set up locally using the Minikube tool. It runs a single-node bunch in a VM on the computer. Therefore, it offers the perfect way for users who have just ongoing learning Kubernetes.

Current options and when to use each:

| Tool | Shape | Good for |
|---|---|---|
| minikube | single-node VM/container | full-feature learning, addons (ingress, metrics-server) |
| kind | Kubernetes in Docker | CI, testing manifests quickly |
| k3d / k3s | lightweight distribution | edge, low-resource labs |
| Docker Desktop | bundled single node | quick local service testing |

```bash
minikube start --cpus=4 --memory=8g
minikube addons enable ingress
minikube addons enable metrics-server
kubectl create namespace data
kubectl apply -f manifests/orders-job.yaml -n data
kubectl get pods -n data -w
```

A local Spark run with `spark-submit --master k8s://...` against minikube is a good way to learn the manifest shape before touching a shared cluster.

minikube runs a node in a VM or container and ships addons such as ingress and metrics-server. kind runs nodes as containers, which is faster to create and easier to run in CI. k3d wraps k3s for a very small footprint. On all three a `LoadBalancer` Service stays `<pending>` unless something provides an implementation: `minikube tunnel`, a cloud-provider addon, or an ingress controller.

Common mistake: testing a `LoadBalancer` Service locally and concluding the manifest is broken. The cloud controller is the missing piece. Use `kubectl port-forward` or the `minikube tunnel` flow instead.

## What is Kubernetes Load Balancing?

Load Balancing is one of the most common and standard ways of exposing the services. There are two types of load balancing in K8s and they are:

Internal load balancer - This type of balancer automatically balances loads and allocates the pods with the required incoming load.

External Load Balancer - This type of balancer directs the traffic from the external loads to backend pods.

Concretely:

+ **Internal**: `ClusterIP` services plus cluster DNS. Pod-to-pod and service-to-service traffic, load balanced by kube-proxy.
+ **External**: `NodePort` (L4, one port per node), `LoadBalancer` (cloud LB), or `Ingress`/Gateway (L7).

```bash
kubectl get svc -n data
kubectl get endpointslices -n data     # the actual pod IPs behind a service
kubectl describe svc warehouse-api -n data
```

For data workloads, "load balancing" also describes distributing work across pods - for example an Airflow Celery queue routed to a worker pool, or Spark task scheduling. Kubernetes service load balancing distributes *connections*, not work items, and a long-lived gRPC/streaming connection can pin to one pod. That is what service meshes and headless services with client-side balancing solve.

Kubernetes balances connections, not requests. kube-proxy picks an endpoint per connection, so short HTTP requests spread well while a long-lived gRPC stream stays pinned to one pod. The fixes are an L7 proxy that balances per request, a headless Service with client-side balancing, or `sessionAffinity` when stickiness is genuinely wanted.

Common mistake: scaling a gRPC service and seeing no traffic shift. All existing streams stay on the original pods. Use an L7 ingress or a service mesh, or have clients re-resolve the headless Service.

## What the following in the Deployment configuration file mean?

Given:

```yaml
spec:
  containers:
    - name: app
      env:
        - name: USER_PASSWORD
          valueFrom:
            secretKeyRef:
              name: some-secret
              key: password
```

Explanation - USER_PASSWORD environment variable will store the value from the password key in the secret called "some-secret" In other words, you reference a value from a Kubernetes Secret.

Related reference styles:

```yaml
env:
  - name: USER_PASSWORD
    valueFrom:
      secretKeyRef:
        name: some-secret
        key: password
        optional: false
  - name: LOG_LEVEL
    valueFrom:
      configMapKeyRef:
        name: app-config
        key: log_level
  - name: POD_NAME
    valueFrom:
      fieldRef:
        fieldPath: metadata.name
```

```bash
kubectl create secret generic some-secret \
  --from-literal=password='...' -n data
```

Caution: environment variables are visible in `kubectl describe pod` output to anyone with read access to the pod (not the value, but `envFrom`/`secretKeyRef` can be misconfigured), are baked into the container process, and do not rotate without a restart. Prefer mounting Secrets as files for credentials that rotate, and prefer workload identity over static keys entirely.

`valueFrom` injects a single key; `envFrom` injects every key from a ConfigMap or Secret. Variables are resolved when the container starts, so a rotated Secret does not reach a running process. A Secret mounted as a volume is different: the kubelet refreshes the projected files, usually within a minute, and most applications pick up the new value without a restart.

Common mistake: putting a credential in a ConfigMap because the app reads configuration from environment variables. Use `secretKeyRef` and, if the tool supports it, mount the secret as a file so rotation works.

## What is the difference between Docker Swarm and Kubernetes?

Below are the main difference between Kubernetes and Docker:

The installation procedure of the K8s is very complicated but if it is once installed then the cluster is robust. On the other hand, the Docker swarm installation process is very simple but the cluster is not at all robust.

Kubernetes can process the auto-scaling but the Docker swarm cannot process the auto-scaling of the pods based on incoming load.

Kubernetes is a full-fledged Framework. Since it maintains the cluster states more consistently so autoscaling is not as fast as Docker Swarm.

| Aspect | Docker Swarm | Kubernetes |
|---|---|---|
| Setup | one command | many components, or a managed control plane |
| Autoscaling | limited | HPA/VPA/cluster autoscaler |
| Ecosystem | Docker-native only | huge: operators, Helm, service mesh, CNI/CSI |
| Stateful workloads | basic | StatefulSets, PDBs, operators |
| Scheduling | simple spread | rich affinity, taints, priorities, preemption |
| Best fit | small/simple deployments | data platforms and multi-tenant clusters |

For data engineering the deciding factor is not raw scheduling but the surrounding ecosystem: Spark's native Kubernetes backend, Airflow's KubernetesExecutor, and the Strimzi/Flink/Trino operators all target the Kubernetes API. That is why Swarm is rarely the answer in a data-platform interview.

The honest interview answer is that Swarm is simpler and Kubernetes is an ecosystem. For data platforms the deciding factor is the API surface: Spark's native backend, Airflow's KubernetesExecutor, KEDA, and the Flink and Strimzi operators all target Kubernetes. Scheduling nuance matters less than the fact that every tool ships a Kubernetes integration first.

Common mistake: comparing only scheduling features. The reason a platform standardizes on Kubernetes is the surrounding tooling and the hiring pool, not the scheduler algorithm.

## How to troubleshoot if the POD is not getting scheduled?

In K8's scheduler is responsible to spawn pods into nodes. There are many factors that can lead to unstartable POD. The most common one is running out of resources, use the commands like `kubectl describe <POD> -n <Namespace>` to see the reason why POD is not started. Also, keep an eye on `kubectl get events` to see all events coming from the cluster.

```bash
kubectl get pods -n data -o wide            # find the Pending pod
kubectl describe pod orders-etl-abc -n data # look at the Events section
kubectl get events -n data --sort-by=.lastTimestamp
kubectl get nodes -o custom-columns=\
NAME:.metadata.name,CPU:.status.allocatable.cpu,MEM:.status.allocatable.memory,TAINTS:.spec.taints
```

Read the event message; it names the cause:

| Event | Meaning | Fix |
|---|---|---|
| `Insufficient cpu` / `memory` | no node has the requested capacity | lower requests, add a node pool, or free capacity |
| `0/N nodes are available ... node(s) had taint` | node selector/toleration mismatch | add the toleration or change the selector |
| `node(s) didn't match Pod's node affinity` | affinity rules exclude all nodes | relax the affinity |
| `volume node affinity conflict` | PVC is bound to another zone | schedule into that zone or use a zonal-scoped claim |
| `exceeded quota` | namespace ResourceQuota hit | raise the quota or free resources |
| `Unschedulable: PodDisruptionBudget` | not scheduling, but eviction blocked | restore replicas or relax the PDB |

Also check the topology spread constraints and the priority class: a low-priority pod waits behind a higher-priority one when the cluster is full.

Scheduling has two phases. Filtering removes nodes that fail hard constraints: insufficient requests, taints without tolerations, affinity mismatch, volume zone conflict, topology spread. Scoring then ranks the survivors by bin-packing, image locality, and spread. If no node survives filtering the pod stays `Pending` and the scheduler records a `FailedScheduling` event. A pod with a lower `priorityClassName` can also be preempted by a higher one.

Common mistake: adding nodes before reading the event. `Insufficient cpu` on every node is a requests problem; `node(s) had taint` is a toleration problem; `didn't match Pod's node affinity` is an affinity problem. The event text is specific.

## How to run a POD on a particular node?

Various methods are available to achieve it.

nodeName: specify the name of a node in POD spec configuration, it will try to run the POD on a specific node.

nodeSelector: Assign a specific label to the node which has special resources and use the same label in POD spec so that POD will run only on that node.

nodeaffinities: required DuringSchedulingIgnoredDuringExecution, preferredDuringSchedulingIgnoredDuringExecution are hard and soft requirements for running the POD on specific nodes. This will be replacing nodeSelector in the future. It depends on the node labels.

```yaml
spec:
  nodeName: ip-10-0-3-17.ec2.internal    # bypasses the scheduler; avoid in production
```

```yaml
spec:
  nodeSelector:
    node-pool: spark
    topology.kubernetes.io/zone: us-east-1a
```

```yaml
spec:
  affinity:
    nodeAffinity:
      requiredDuringSchedulingIgnoredDuringExecution:
        nodeSelectorTerms:
          - matchExpressions:
              - key: node-pool
                operator: In
                values: ["spark"]
      preferredDuringSchedulingIgnoredDuringExecution:
        - weight: 80
          preference:
            matchExpressions:
              - key: lifecycle
                operator: In
                values: ["spot"]
    podAntiAffinity:
      preferredDuringSchedulingIgnoredDuringExecution:
        - weight: 100
          podAffinityTerm:
            topologyKey: kubernetes.io/hostname
            labelSelector:
              matchLabels: {app: kafka}
```

For data workloads, `nodeSelector` for a GPU pool plus a toleration for the pool's taint is the standard pattern; a Spark executor that needs an instance type should express it as affinity, not as a hard `nodeName`, so the scheduler can still pack and scale the pool.

`nodeSelector` is a hard equality match on one or more labels. `nodeAffinity` adds operators, multiple terms, and a soft `preferred` form that the scorer weights. `nodeName` skips the scheduler entirely, so no taint or resource check happens and the pod may be placed where it cannot run. For a fleet, prefer affinity: the scheduler can still pack, preempt, and spread.

Common mistake: using `nodeName` in a template that is scaled. New replicas all target one node, ignore taints, and overload it while other nodes idle.

## What are the different ways to provide external network connectivity to K8?

By default, POD should be able to reach the external network but vice-versa we need to make some changes. Following options are available to connect with POD from the outer world.

Nodeport (it will expose one port on each node to communicate with it)

Load balancers (L4 layer of TCP/IP protocol)

Ingress (L7 layer of TCP/IP Protocol)

Another method is to use Kube-proxy which can expose a service with only cluster IP on the local system port.

```bash
kubectl proxy --port=8080
# then: http://localhost:8080/api/v1/namespaces/data/services/warehouse-api:http/proxy/
```

Two more options worth naming: **Gateway API** (the successor to Ingress, with richer routing and multi-team ownership) and **port-forward** for debugging only:

```bash
kubectl port-forward svc/warehouse-api 8080:80 -n data
```

For egress to the outside world, the pattern is the reverse: a NAT gateway, an egress gateway with a static IP (when a vendor allow-lists an IP), or a NetworkPolicy that permits only the destinations a job needs.

`NodePort` allocates from the default range 30000-32767 on every node. `LoadBalancer` delegates to the cloud controller and accepts provider annotations such as an internal scheme or a static IP. `Ingress` and the newer Gateway API are L7 and share one address across many services. For pods that need to reach the internet through a fixed address, the mirror image is an egress gateway or NAT with a stable IP.

Common mistake: assuming a `LoadBalancer` Service is private. Unless a provider annotation marks it internal, it gets a public address by default.

## How can we forward the port 8080 container to 8080 service to 8080 ingress to 80 browser and how it can be done?

The ingress is exposing port 80 externally for the browser to access, and connecting to a service that listens on 8080. The ingress will listen on port 80 by default. An "ingress controller" is a pod that receives external traffic and handles the ingress and is configured by an ingress resource. For this you need to configure the ingress selector and if no 'ingress controller selector' is mentioned then no ingress controller will manage the ingress.

Simple ingress config will look like:

```yaml
apiVersion: networking.k8s.io/v1
kind: Ingress
metadata:
  name: abc-ingress
spec:
  ingressClassName: nginx
  rules:
    - host: abc.org
      http:
        paths:
          - path: /
            pathType: Prefix
            backend:
              service:
                name: abc-service
                port:
                  number: 8080
```

Then the service will look like:

```yaml
apiVersion: v1
kind: Service
metadata:
  name: abc-service
spec:
  selector:
    app: abc
  ports:
    - protocol: TCP
      port: 8080          # port the service listens on
      targetPort: 8080    # port the container listens on
```

Finally, the container declares the same port:

```yaml
spec:
  containers:
    - name: abc
      image: registry.example.com/data/abc:1.0.0
      ports:
        - containerPort: 8080
```

The chain in one line: browser `http://abc.org:80` -> ingress controller (80) -> Service `abc-service:8080` -> pod endpoint `:8080` -> container process. Each hop has its own port number and they do not have to match; the `targetPort` is what ties the service to the container. If the browser times out, check in order: DNS resolves, the ingress has an external address, the ingress controller is running, the Service has endpoints, and the container is actually listening.

`kubectl port-forward` opens a tunnel through the API server to the kubelet and then to the pod, which is why it works even when no Service exists. It binds to localhost by default and dies with the terminal. It is a debugging tool: one connection, no load balancing, and no authentication beyond your kubeconfig.

Common mistake: leaving a port-forward running as the integration path. Anything that must be reachable by another service or a browser session needs a Service and an Ingress, not a tunnel from one laptop.

## How do you configure liveness, readiness, and startup probes?

Probes are how Kubernetes decides whether a container is alive and whether it should receive traffic. Getting them wrong is a classic cause of cascading restarts.

+ **readinessProbe** - gates traffic. A failing readiness probe removes the pod from the Service endpoints but leaves it running.
+ **livenessProbe** - restarts the container when it is stuck (deadlock, exhausted thread pool). It must not fail during normal slow operation.
+ **startupProbe** - protects a slow-starting container; until it succeeds, liveness and readiness are disabled.

```yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: warehouse-api
  namespace: data
spec:
  replicas: 3
  selector:
    matchLabels: {app: warehouse-api}
  template:
    metadata:
      labels: {app: warehouse-api}
    spec:
      containers:
        - name: api
          image: registry.example.com/data/warehouse-api:2.3.0
          ports:
            - containerPort: 8080
          startupProbe:
            httpGet: {path: /healthz, port: 8080}
            failureThreshold: 30
            periodSeconds: 10          # up to 5 minutes to start
          readinessProbe:
            httpGet: {path: /ready, port: 8080}
            initialDelaySeconds: 5
            periodSeconds: 10
            timeoutSeconds: 2
            failureThreshold: 3
          livenessProbe:
            httpGet: {path: /healthz, port: 8080}
            periodSeconds: 20
            timeoutSeconds: 3
            failureThreshold: 3
```

Guidance that prevents incidents:

+ A liveness probe should check the process is responsive, not that a downstream dependency is up. If the liveness probe fails when the warehouse is briefly unreachable, Kubernetes restarts every pod and the outage gets worse.
+ Readiness is where you check dependencies: the API is not ready to serve until its metadata database is reachable.
+ Use `exec` probes only when HTTP/TCP cannot express the check; a shell-based probe needs the binary inside the image and is easy to get wrong.
+ For batch Jobs, probes are usually unnecessary: the process either exits 0 or it does not.

The kubelet runs probes on a schedule and tracks success and failure counts. `initialDelaySeconds` waits before the first check, `periodSeconds` sets the interval, `timeoutSeconds` bounds one check, and `failureThreshold` how many failures trigger the action. A failing readiness probe removes the pod from EndpointSlices; a failing liveness probe kills and restarts the container; a startup probe gates the other two so a slow JVM boot does not start a restart loop.

Common mistake: pointing the liveness probe at a dependency. A warehouse blip then fails the probe on every replica at once and Kubernetes restarts a healthy fleet. Check dependencies in readiness, and keep liveness about the process itself.

## How do you use node selectors, affinity, and tolerations?

Selectors and affinity attract a pod to a node; taints and tolerations are the node-side counterpart that repels pods unless they explicitly tolerate the taint.

```bash
# label a dedicated Spark pool, then taint it so only Spark jobs land there
kubectl label node ip-10-0-3-17 node-pool=spark
kubectl taint node ip-10-0-3-17 workload=spark:NoSchedule
```

```yaml
spec:
  nodeSelector:
    node-pool: spark
  tolerations:
    - key: workload
      operator: Equal
      value: spark
      effect: NoSchedule
  containers:
    - name: spark-executor
      image: registry.example.com/data/spark:3.5.3
```

Choose the mechanism deliberately:

| Mechanism | Strength | Use for |
|---|---|---|
| `nodeSelector` | hard, exact label match | simple pool placement |
| `nodeAffinity` required | hard, expressive | instance type / zone / GPU |
| `nodeAffinity` preferred | soft | cost preference (spot over on-demand) |
| `podAntiAffinity` | avoids co-location | spread brokers/executors across nodes |
| `topologySpreadConstraints` | even spread across zones | multi-AZ resilience |
| taints + tolerations | reserves nodes | dedicated Spark, GPU, or licensing pools |

```yaml
# spread executor pods evenly across zones
topologySpreadConstraints:
  - maxSkew: 1
    topologyKey: topology.kubernetes.io/zone
    whenUnsatisfiable: ScheduleAnyway
    labelSelector:
      matchLabels: {app: spark-executor}
```

The classic interview answer: use a taint on a dedicated pool so no random pod consumes it, give your Spark/Airflow pods the matching toleration, and use pod anti-affinity to avoid all executors on one node.

Taints repel, tolerations permit. `NoSchedule` keeps new pods off a node, `PreferNoSchedule` is a soft hint, and `NoExecute` also evicts running pods that do not tolerate it. A toleration without a matching nodeSelector only grants permission; it does not attract the pod to that pool. Built-in taints include `node.kubernetes.io/not-ready`, `node.kubernetes.io/unreachable`, and the control-plane taint.

Common mistake: adding a toleration and expecting the pod to land on the tainted pool. It may land anywhere. Add the matching `nodeSelector` or required node affinity as well.

## How do you autoscale a data workload with the Horizontal Pod Autoscaler?

The HPA scales the replica count of a Deployment or StatefulSet (and, with newer APIs, custom resources) based on metrics. It needs the metrics-server or a custom-metrics adapter.

```yaml
apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata:
  name: warehouse-api
  namespace: data
spec:
  scaleTargetRef:
    apiVersion: apps/v1
    kind: Deployment
    name: warehouse-api
  minReplicas: 3
  maxReplicas: 30
  behavior:
    scaleDown:
      stabilizationWindowSeconds: 300
      policies:
        - type: Percent
          value: 50
          periodSeconds: 60
    scaleUp:
      stabilizationWindowSeconds: 30
  metrics:
    - type: Resource
      resource:
        name: cpu
        target:
          type: Utilization
          averageUtilization: 65
    - type: Pods
      pods:
        metric: {name: kafka_consumer_lag}
        target:
          type: AverageValue
          averageValue: "10000"
```

Important limitations for data engineering:

+ The HPA does **not** replace the cluster autoscaler. If no node has capacity, new pods stay Pending; the cluster autoscaler (or Karpenter) adds nodes.
+ The HPA is poorly suited to batch Jobs - a Job already declares its parallelism, and a run is not a long-lived service. Scale batch work with `spark.executor.instances`, dynamic allocation, or job queueing instead.
+ Queue-driven scaling (Kafka lag, SQS depth, Airflow queue length) is the right trigger for streaming consumers and worker pools; CPU utilization alone lags the real backlog.
+ Always set `minReplicas` above one for a tier-1 service and a `stabilizationWindowSeconds` long enough to avoid flapping.

The controller reads the metric every sync period (15 seconds by default), computes `desired = ceil(currentReplicas * currentMetric / targetMetric)`, and ignores changes within a 10 percent tolerance. Resource metrics are utilization of requests, so a container with no CPU request reports `<unknown>` and the HPA does nothing. `behavior` controls how fast it may move.

Common mistake: an HPA that reports `<unknown>` or does not scale because the target Deployment has no `resources.requests.cpu`. Utilization needs a denominator. Set requests before you turn on the autoscaler.

## How do you run Spark on Kubernetes?

Spark has a native Kubernetes backend: `spark-submit` talks to the Kubernetes API server, creates a driver pod, and the driver creates executor pods. There is no Spark master or worker process to manage.

```bash
spark-submit \
  --master k8s://https://<api-server>:443 \
  --deploy-mode cluster \
  --name orders-daily \
  --conf spark.kubernetes.namespace=data \
  --conf spark.kubernetes.container.image=registry.example.com/data/spark:3.5.3 \
  --conf spark.kubernetes.authenticate.driver.serviceAccountName=spark \
  --conf spark.executor.instances=20 \
  --conf spark.executor.cores=4 \
  --conf spark.executor.memory=12g \
  --conf spark.driver.cores=2 \
  --conf spark.driver.memory=4g \
  --conf spark.kubernetes.executor.request.cores=4 \
  --conf spark.kubernetes.executor.limit.cores=6 \
  --conf spark.dynamicAllocation.enabled=true \
  --conf spark.dynamicAllocation.minExecutors=2 \
  --conf spark.dynamicAllocation.maxExecutors=60 \
  --conf spark.kubernetes.executor.deleteOnTermination=true \
  --conf spark.kubernetes.driver.podTemplateFile=driver-pod-template.yaml \
  --conf spark.kubernetes.executor.podTemplateFile=executor-pod-template.yaml \
  s3://acme-code/jobs/orders_daily.py
```

A pod template is how you attach node selectors, tolerations, volumes, and sidecars to the driver and executors:

```yaml
apiVersion: v1
kind: Pod
metadata:
  labels:
    app: spark
spec:
  nodeSelector:
    node-pool: spark
  tolerations:
    - key: workload
      operator: Equal
      value: spark
      effect: NoSchedule
  containers:
    - name: spark-kubernetes-executor
      resources:
        requests:
          cpu: "4"
          memory: "16Gi"
        limits:
          cpu: "6"
          memory: "18Gi"
      volumeMounts:
        - name: shuffle
          mountPath: /tmp/spark-local
  volumes:
    - name: shuffle
      emptyDir:
        sizeLimit: 100Gi
```

Operational reality:

+ **Use cluster mode**, not client mode, for production: the driver runs in-cluster and survives your laptop disconnecting.
+ **`spark.kubernetes.executor.request.cores` is the request; `spark.executor.cores` must not exceed it**, or executors get throttled and shuffle throughput collapses.
+ Executors are separate pods, so a node failure kills several executors at once; Spark retries the stages, but shuffle data on `emptyDir` is lost.
+ Pod startup adds 10-60 seconds per executor. For short jobs the overhead dominates; for the same workload check whether a managed runtime (Databricks, EMR, Dataproc) is cheaper in engineering time.
+ Use image pull secrets and pre-pulled images on the node pool to avoid the startup penalty, and set `spark.kubernetes.executor.deleteOnTermination` deliberately.

```yaml
apiVersion: sparkoperator.k8s.io/v1beta2
kind: SparkApplication
metadata:
  name: orders-daily
  namespace: data
spec:
  type: Python
  pythonVersion: "3"
  mode: cluster
  image: registry.example.com/data/spark:3.5.3
  mainApplicationFile: s3://acme-code/jobs/orders_daily.py
  sparkVersion: 3.5.3
  restartPolicy:
    type: OnFailure
    onFailureRetries: 3
    onFailureRetryInterval: 30
  driver:
    cores: 2
    coreLimit: "3"
    memory: 4g
    serviceAccount: spark
    labels: {app: spark-driver}
  executor:
    cores: 4
    coreLimit: "6"
    instances: 20
    memory: 12g
    labels: {app: spark-executor}
```

The driver is a pod that talks to the API server, and each executor is another pod the driver creates. Dynamic allocation adds and removes executors as stages demand them, up to `spark.dynamicAllocation.maxExecutors`. Executors share a node, so a node failure loses several at once together with their shuffle output; Spark reruns the affected stages.

Common mistake: setting `spark.executor.cores` above `spark.kubernetes.executor.request.cores`. The scheduler packs pods by the request while the JVM uses the larger core count, so two executors can land on a node sized for one and both thrash. Keep the request at or above the cores Spark will use.

## How do you run Airflow workers on Kubernetes?

Two patterns coexist. With `KubernetesExecutor`, each task gets its own pod (or the whole DAG, with `pod_override`). With `CeleryKubernetesExecutor` (Airflow 2) or multi-executor config (Airflow 3), long-running workers are Celery pods and only heavy tasks go to Kubernetes.

Worker deployment:

```yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: airflow-worker
  namespace: airflow
spec:
  replicas: 4
  selector:
    matchLabels: {app: airflow-worker}
  template:
    metadata:
      labels: {app: airflow-worker}
    spec:
      serviceAccountName: airflow-worker
      terminationGracePeriodSeconds: 120
      containers:
        - name: worker
          image: registry.example.com/data/airflow:3.3.0
          command: ["airflow", "celery", "worker"]
          env:
            - name: AIRFLOW__CELERY__BROKER_URL
              valueFrom:
                secretKeyRef: {name: airflow-broker, key: url}
            - name: AIRFLOW__DATABASE__SQL_ALCHEMY_CONN
              valueFrom:
                secretKeyRef: {name: airflow-metadata, key: url}
          resources:
            requests: {cpu: "1", memory: 2Gi}
            limits: {cpu: "2", memory: 4Gi}
          readinessProbe:
            exec:
              command: ["airflow", "jobs", "check", "--job-type", "CeleryWorker"]
            initialDelaySeconds: 30
            periodSeconds: 30
```

Per-task pod template for KubernetesExecutor:

```yaml
apiVersion: v1
kind: Pod
metadata:
  name: placeholder
spec:
  serviceAccountName: airflow-worker
  nodeSelector:
    node-pool: general
  containers:
    - name: base
      image: registry.example.com/data/airflow:3.3.0
      resources:
        requests: {cpu: "500m", memory: 1Gi}
        limits: {cpu: "2", memory: 4Gi}
  restartPolicy: Never
```

Guidance:

+ Set the worker `terminationGracePeriodSeconds` longer than the longest task, and use `[celery] worker_autoscale` or KEDA to scale on queue depth.
+ Give the pod template a service account that can only create pods in the Airflow namespace - `KubernetesExecutor` needs pod create/delete permissions, and a wildcard role there is a real privilege-escalation risk.
+ Mount `dags/`, `logs/`, and `plugins/` from object storage or a PVC; the scheduler and workers must see the same DAG files and log directory, or tasks fail with import errors.
+ Log to stdout and ship to the central stack; `remote_logging` to S3/GCS is the alternative and is what makes task logs survive pod deletion.

`KubernetesExecutor` creates one pod per task, so each task can carry its own image, resources, and service account. It scales to zero when idle and pays 10-30 seconds of pod startup per task. Celery keeps warm workers and suits many short tasks; the combined executor routes only heavy tasks to pods. The worker pod template and `pod_override` decide resources per task.

Common mistake: giving the executor service account cluster-wide pod creation. It only needs to create, watch, and delete pods in the Airflow namespace. A wildcard role there is a privilege-escalation path.

## How do you manage secrets and configuration for data workloads?

ConfigMaps hold non-sensitive configuration; Secrets hold credentials, but a Kubernetes Secret is only base64-encoded by default, so encryption at rest and RBAC matter.

```yaml
apiVersion: v1
kind: ConfigMap
metadata:
  name: orders-etl-config
  namespace: data
data:
  LOG_LEVEL: "INFO"
  TARGET_TABLE: "analytics.orders"
  spark-defaults.conf: |
    spark.sql.shuffle.partitions=400
    spark.sql.adaptive.enabled=true
---
apiVersion: v1
kind: Secret
metadata:
  name: warehouse-credentials
  namespace: data
type: Opaque
stringData:
  username: etl_user
  password: "REPLACE_WITH_SECRET"
```

Mount configuration as a file when the tool expects a config file, and as env vars only for simple scalars:

```yaml
spec:
  containers:
    - name: orders-etl
      image: registry.example.com/data/orders-etl:1.14.2
      envFrom:
        - configMapRef: {name: orders-etl-config}
        - secretRef: {name: warehouse-credentials}
      volumeMounts:
        - name: spark-config
          mountPath: /opt/spark/conf
          readOnly: true
  volumes:
    - name: spark-config
      configMap: {name: orders-etl-config}
```

Best practice in order of preference:

1. **Workload identity / IRSA / Workload Identity Federation** - the pod assumes a cloud role and gets short-lived credentials. No static secret at all.
2. **External secret manager** (Vault, AWS Secrets Manager, GCP Secret Manager) synced into a Secret by an operator, with rotation.
3. **Sealed/encrypted GitOps secrets** (Sealed Secrets, SOPS) so the ciphertext can live in git.
4. Plain `kubectl create secret` - acceptable for a lab, not for a multi-team cluster.

Also restrict access with RBAC (`get` on Secrets is powerful), enable encryption at rest with a KMS key, and never put a credential in a ConfigMap or an image layer.

Secrets come in types: `Opaque` for key-value pairs, `kubernetes.io/tls` for certificates, `kubernetes.io/dockerconfigjson` for registry credentials, and the legacy service-account token type. `immutable: true` stops accidental edits and reduces kubelet watch load. A projected service account token with an audience and an expiry is the modern replacement for long-lived tokens.

Common mistake: treating base64 as encryption. Anyone with `get secrets` in the namespace can read the value. Enable encryption at rest with a KMS key, restrict RBAC, and prefer workload identity so there is no static credential to read.

## How do you run a scheduled data pipeline with a CronJob?

A CronJob is the Kubernetes-native scheduler for a containerized batch job. It is a good fit for self-contained jobs (an extract, a compaction, a small load) and a poor fit for pipelines with many dependent steps - that is what Airflow, Dagster, or Argo Workflows are for.

```yaml
apiVersion: batch/v1
kind: CronJob
metadata:
  name: orders-hourly-load
  namespace: data
spec:
  schedule: "15 * * * *"
  timeZone: "UTC"
  concurrencyPolicy: Forbid
  suspend: false
  startingDeadlineSeconds: 300
  successfulJobsHistoryLimit: 3
  failedJobsHistoryLimit: 5
  jobTemplate:
    spec:
      activeDeadlineSeconds: 5400
      backoffLimit: 2
      ttlSecondsAfterFinished: 172800
      template:
        spec:
          restartPolicy: OnFailure
          serviceAccountName: orders-loader
          containers:
            - name: load
              image: registry.example.com/data/orders-loader:1.7.0
              args:
                - "--window-start"
                - "$(date -u -d '1 hour ago' +%Y-%m-%dT%H:00:00Z)"
              resources:
                requests: {cpu: "1", memory: 2Gi}
                limits: {cpu: "2", memory: 4Gi}
              envFrom:
                - secretRef: {name: warehouse-credentials}
```

Fields that matter:

+ `concurrencyPolicy: Forbid` prevents a slow run from overlapping the next one - essential when both write the same target.
+ `startingDeadlineSeconds` + `activeDeadlineSeconds` bound a stuck job instead of letting it run for days.
+ `suspend: true` is the kill switch during an incident, without deleting the object.
+ `ttlSecondsAfterFinished` cleans up completed Job objects so the namespace does not accumulate thousands of them.
+ Use `schedule` in UTC unless the business window is genuinely in a local timezone; `timeZone` exists but timezone-aware schedules surprise people during DST changes.

For anything with more than two or three dependent steps, put the orchestration in Airflow and let Kubernetes run the tasks, rather than encoding dependencies with multiple CronJobs.

A CronJob records the last schedule time and creates a Job for the next one. If the controller is down past `startingDeadlineSeconds` the run is counted as missed; more than 100 missed schedules produce a `TooManyMissedTimes` event and no job. `concurrencyPolicy: Allow` runs them together, `Forbid` skips the new one, and `Replace` cancels the running one. Inside the Job, `backoffLimit` counts pod retries and `activeDeadlineSeconds` bounds the whole run.

Common mistake: `concurrencyPolicy: Allow` on a job that writes the same target. A slow run and the next run interleave writes and produce duplicates or corrupt partitions. Use `Forbid`, or make the target write idempotent with a partition overwrite and a run id.

## How do you right-size requests and limits for data workloads?

Right-sizing starts from measurement. `kubectl top pod` and cAdvisor give current usage, but batch jobs peak in bursts, so use the peak over the run rather than the average. A container that peaks at 6 GiB and requests 8 GiB is safe; one that peaks at 6 GiB and requests 2 GiB is a candidate for eviction the moment the node comes under pressure.

The two failure modes are asymmetric. Too small a CPU limit throttles the container: the CFS quota stops it mid-period even on an idle node, and a JVM or shuffle-heavy operator slows down quietly. Too small a memory limit does not throttle, it kills: the cgroup OOM killer ends the process, the kubelet reports `OOMKilled`, and the container restarts with exit code 137. Set memory to cover the real peak, and set CPU requests for packing while leaving CPU headroom, or no CPU limit, for bursty work.

For a JVM container the heap must fit inside the limit with room for metaspace, thread stacks, code cache, and direct buffers. Set `-XX:MaxRAMPercentage` and let the JVM derive the heap from the container limit instead of hard-coding both.

```yaml
resources:
  requests:
    cpu: "4"
    memory: "16Gi"
  limits:
    memory: "16Gi"
env:
  - name: JAVA_TOOL_OPTIONS
    value: "-XX:MaxRAMPercentage=75 -XX:InitialRAMPercentage=50 -XX:+UseG1GC"
```

Common mistake: setting a memory limit equal to `spark.executor.memory`. Spark's `memoryOverhead` (by default `max(384Mi, 10% of executor memory)`) covers off-heap work, and the JVM adds its own overhead on top. A 12g heap behind a 12Gi limit is OOM-killed under load. Size the limit from the heap plus overhead, then verify with `kubectl top` and the restart count.

## How do you autoscale pods and nodes for data workloads?

The HPA scales replicas from metrics; the cluster autoscaler or Karpenter scales nodes from pending pods. They are a pair: the HPA adds pods, and if no node can fit them the pods stay `Pending` until a node appears. Neither reacts instantly. The HPA syncs every 15 seconds by default and honours a stabilization window; a new node takes a minute or more to join and pull images.

The Vertical Pod Autoscaler changes requests and limits instead of replica count. In `Off` mode it only recommends, which is where to start. `Initial` sets them at pod creation, and `Recreate` or `Auto` evicts and recreates pods to apply new values, which is disruptive for stateful workloads. Do not let a VPA and an HPA manage the same CPU or memory metric on the same workload; they will fight.

```yaml
apiVersion: autoscaling.k8s.io/v1
kind: VerticalPodAutoscaler
metadata:
  name: catalog-api
  namespace: data
spec:
  targetRef:
    apiVersion: apps/v1
    kind: Deployment
    name: catalog-api
  updatePolicy:
    updateMode: "Off"
```

The cluster autoscaler grows a node group when a pending pod fits its instance type and shrinks idle groups. Karpenter provisions instances directly from the pods' requirements, so it can pick a size that fits and consolidate later, without pre-created groups. A practical pool mix is on-demand nodes for drivers, schedulers, and stateful services, and spot nodes for executors and workers, separated by a taint so only tolerant pods land there.

Common mistake: an HPA pinned at `maxReplicas` with pending pods. That is a cluster-capacity problem, not a pod-scaling problem. Check `kubectl get pods` for `Pending`, the autoscaler events, and whether the pod requests are larger than any node type the autoscaler is allowed to create.

## How do you schedule data workloads across node pools?

A node pool is a set of nodes with the same instance type, labels, and taints. Give each pool a label such as `node-pool: spark` and a taint such as `workload=spark:NoSchedule`, then have the executor pod template carry both the selector and the toleration. That reserves the pool for Spark without letting a stray pod consume it.

```yaml
spec:
  nodeSelector:
    node-pool: spark
    capacity-type: spot
  tolerations:
    - key: workload
      operator: Equal
      value: spark
      effect: NoSchedule
  topologySpreadConstraints:
    - maxSkew: 1
      topologyKey: topology.kubernetes.io/zone
      whenUnsatisfiable: ScheduleAnyway
      labelSelector:
        matchLabels: {app: spark-executor}
  priorityClassName: batch-normal
```

Spark needs many executors at once. If only a few schedule, the job waits for the rest while holding the nodes it already got. `spark.scheduler.minRegisteredResourcesRatio` and `spark.scheduler.maxRegisteredResourcesWaitingTime` bound that wait. True all-or-nothing is gang scheduling, which operators such as Volcano and YuniKorn provide with a PodGroup: either enough pods are admitted together, or none are.

Common mistake: a toleration with no nodeSelector. The pod is allowed on the tainted pool but also everywhere else, so the dedicated pool stays empty and the pod competes with interactive services. Pair every taint with an explicit selector.

## How do you debug slow or failing pods?

Start with state, then events, then logs. `kubectl get pods` shows phase and restart count; `kubectl describe pod` shows the events, the node, and the last terminated state; `kubectl logs --previous` shows the crashed container; `kubectl top pod` shows whether it is resource-bound. The exit code in `describe` narrows it down: 137 is SIGKILL or OOMKilled, 1 is an application error, and 0 is a clean finish.

| Symptom | Usual cause | First check |
|---|---|---|
| `Pending` | no node fits the requests, taints, or affinity | `kubectl describe pod` events |
| `ContainerCreating` | image pull, volume mount, or CNI | events, image pull secrets |
| `ImagePullBackOff` | wrong tag or missing registry credentials | image name, `imagePullSecrets` |
| `CrashLoopBackOff` | the process exits immediately | `kubectl logs --previous` |
| `OOMKilled` (137) | memory limit below the peak | `kubectl top pod`, limit |
| `Evicted` | node memory or disk pressure | node conditions, QoS class |
| `Error` / `Failed` | non-zero exit after retries | Job `backoffLimit`, logs |
| Slow, not failing | CPU throttling or a starved dependency | throttled seconds, `kubectl top` |

Common mistake: restarting the workload first and reading the state second. A `kubectl rollout restart` erases the failing pod and its previous logs with it. Capture `describe` and `logs --previous` before you change anything.

## Kubernetes exercises

These exercises use only the Python standard library and model the sizing and scheduling arithmetic behind the manifest fields. Each names the task, gives a hint, states the expected result, and ends in a runnable cell. Run them with `/Users/sai/WS/data-engineering-interview-questions/.venv/bin/python`.

The numbers mirror the units Kubernetes uses: CPU in cores or millicores, memory in MiB and GiB, utilization as a percentage of requests, and replica counts rounded up. Treat the asserted values as a check on the formula, then verify against a real cluster with `kubectl top` and `kubectl describe`.

### Try it 1. Size a JVM Spark executor

**Task.** An executor needs a 12 GiB heap and runs with `-XX:MaxRAMPercentage=75`. Compute the container memory limit so the heap fits. Also compute Spark's default `memoryOverhead` and the limit you would choose without `MaxRAMPercentage`.

**Hint.** With `MaxRAMPercentage` the heap is a fraction of the limit, so `limit = heap / fraction`, rounded up to a whole GiB. Without it, `limit = heap + overhead`, rounded up.

**Expected result.** The `MaxRAMPercentage` limit is `16 GiB` because `12 / 0.75 = 16`. The default overhead is `1228 MiB` and the heap-plus-overhead limit is `14 GiB`. The heap is `75%` of the limit.

In [ ]:
import math

HEAP_GIB = 12
MAX_RAM_PERCENTAGE = 0.75
OVERHEAD_FLOOR_MIB = 384
OVERHEAD_FRACTION = 0.10
GIB_IN_MIB = 1024


def limit_with_max_ram(heap_gib, fraction):
    return math.ceil(heap_gib / fraction)


def overhead_mib(heap_gib):
    return max(OVERHEAD_FLOOR_MIB, int(heap_gib * GIB_IN_MIB * OVERHEAD_FRACTION))


def limit_with_overhead(heap_gib):
    total_mib = heap_gib * GIB_IN_MIB + overhead_mib(heap_gib)
    return math.ceil(total_mib / GIB_IN_MIB)


max_ram_limit = limit_with_max_ram(HEAP_GIB, MAX_RAM_PERCENTAGE)
overhead = overhead_mib(HEAP_GIB)
overhead_limit = limit_with_overhead(HEAP_GIB)
heap_fraction = HEAP_GIB / max_ram_limit

print(f"MaxRAMPercentage limit = {max_ram_limit} GiB")
print(f"default memoryOverhead = {overhead} MiB")
print(f"heap-plus-overhead limit = {overhead_limit} GiB")
print(f"heap is {heap_fraction:.0%} of the limit")

assert max_ram_limit == 16
assert overhead == 1228
assert overhead_limit == 14
assert heap_fraction == 0.75

### Try it 2. Compute HPA replicas from a CPU target

**Task.** Given current replicas and current CPU utilization, compute the replica count the HPA wants, using `desired = ceil(current * utilization / target)` and ignoring changes inside a 10 percent tolerance. Apply it to 4 replicas at 90 percent with a 60 percent target, and to 10 replicas at 63 percent with the same target.

**Hint.** Compute `ratio = utilization / target`. If `abs(ratio - 1) <= 0.1`, keep the current count. Otherwise round the product up and clamp to `minReplicas` and `maxReplicas`.

**Expected result.** The first case scales to `6`; the second stays at `10` because 1.05 is inside the tolerance.

In [ ]:
import math

TOLERANCE = 0.1


def hpa_desired(current, utilization, target, min_replicas=1, max_replicas=30):
    ratio = utilization / target
    if abs(ratio - 1.0) <= TOLERANCE:
        return current
    desired = math.ceil(current * ratio)
    return max(min_replicas, min(max_replicas, desired))


cases = [(4, 90, 60), (10, 63, 60)]
for current, utilization, target in cases:
    print(f"{current} replicas at {utilization}% toward {target}% -> {hpa_desired(current, utilization, target)}")

assert hpa_desired(4, 90, 60) == 6
assert hpa_desired(10, 63, 60) == 10
assert hpa_desired(2, 0, 60) == 1

### Try it 3. Detect CPU throttling from container metrics

**Task.** From cAdvisor counters `container_cpu_cfs_periods_total` and `container_cpu_cfs_throttled_periods_total`, compute the throttled fraction for each container and flag any above 25 percent.

**Hint.** `fraction = throttled / periods`, and guard against a zero denominator.

**Expected result.** `spark-executor` at `38.0%` and `dbt` at `41.7%` are flagged; `api` at `2.5%` is fine.

In [ ]:
THROTTLE_THRESHOLD = 0.25

samples = {
    # container: (cfs_periods_total, cfs_throttled_periods_total)
    "api": (48_000, 1_200),
    "spark-executor": (200_000, 76_000),
    "dbt": (12_000, 5_000),
}


def throttled_fraction(periods, throttled):
    if periods == 0:
        return 0.0
    return throttled / periods


flagged = []
for name, (periods, throttled) in samples.items():
    fraction = throttled_fraction(periods, throttled)
    status = "throttled" if fraction > THROTTLE_THRESHOLD else "ok"
    print(f"{name:<15} {fraction:6.1%}  {status}")
    if status == "throttled":
        flagged.append(name)

assert round(throttled_fraction(48_000, 1_200), 4) == 0.025
assert round(throttled_fraction(200_000, 76_000), 4) == 0.38
assert round(throttled_fraction(12_000, 5_000), 4) == 0.4167
assert flagged == ["spark-executor", "dbt"]

### Try it 4. Choose a QoS class from requests versus limits

**Task.** Classify each container spec as `Guaranteed`, `Burstable`, or `BestEffort`. A pod is `Guaranteed` when every container sets CPU and memory requests equal to their limits, `BestEffort` when no container sets any CPU or memory request or limit, and `Burstable` otherwise.

**Hint.** Walk every container and resource. Track whether anything was set at all, and whether every request matched its limit.

**Expected result.** The classes are `Burstable`, `Guaranteed`, `BestEffort`, and `Burstable`.

In [ ]:
def qos_class(containers):
    anything_set = False
    all_guaranteed = True
    for container in containers:
        requests = container.get("requests") or {}
        limits = container.get("limits") or {}
        for resource in ("cpu", "memory"):
            if resource in requests or resource in limits:
                anything_set = True
            if resource not in requests or resource not in limits or requests[resource] != limits[resource]:
                all_guaranteed = False
    if not anything_set:
        return "BestEffort"
    return "Guaranteed" if all_guaranteed else "Burstable"


specs = {
    "api": [{"requests": {"cpu": "500m", "memory": "1Gi"}, "limits": {"cpu": "2", "memory": "4Gi"}}],
    "spark-driver": [{"requests": {"cpu": "2", "memory": "8Gi"}, "limits": {"cpu": "2", "memory": "8Gi"}}],
    "scratch": [{}],
    "sidecar": [{"requests": {"memory": "1Gi"}, "limits": {"memory": "2Gi"}}],
}

classes = {}
for name, containers in specs.items():
    classes[name] = qos_class(containers)
    print(f"{name:<13} {classes[name]}")

assert classes["api"] == "Burstable"
assert classes["spark-driver"] == "Guaranteed"
assert classes["scratch"] == "BestEffort"
assert classes["sidecar"] == "Burstable"
assert list(classes.values()) == ["Burstable", "Guaranteed", "BestEffort", "Burstable"]

### Try it 5. Compute node count from pod needs and headroom

**Task.** A backfill has 100 executor pods, each requesting 4 CPU and 16 GiB. Each node allocates 32 CPU and 128 GiB, and 15 percent is reserved for system and DaemonSet pods. Compute the pods per node and the nodes needed.

**Hint.** `usable = allocatable * (1 - headroom)`. Then `pods_per_node = min(floor(usable_cpu / pod_cpu), floor(usable_mem / pod_mem))` and `nodes = ceil(total / pods_per_node)`.

**Expected result.** Usable per node is `27.2 CPU` and `108.8 GiB`, so 6 pods fit per node and 100 executors need `17` nodes.

In [ ]:
import math

TOTAL_PODS = 100
POD_CPU = 4
POD_MEM_GIB = 16
NODE_CPU = 32
NODE_MEM_GIB = 128
HEADROOM = 0.15

usable_cpu = NODE_CPU * (1 - HEADROOM)
usable_mem = NODE_MEM_GIB * (1 - HEADROOM)
pods_by_cpu = math.floor(usable_cpu / POD_CPU)
pods_by_mem = math.floor(usable_mem / POD_MEM_GIB)
pods_per_node = min(pods_by_cpu, pods_by_mem)
nodes = math.ceil(TOTAL_PODS / pods_per_node)

print(f"usable per node = {usable_cpu:.1f} CPU, {usable_mem:.1f} GiB")
print(f"pods per node = min({pods_by_cpu}, {pods_by_mem}) = {pods_per_node}")
print(f"nodes for {TOTAL_PODS} executors = {nodes}")

assert pods_by_cpu == 6
assert pods_by_mem == 6
assert pods_per_node == 6
assert nodes == 17

### Try it 6. Decide eviction risk from memory usage

**Task.** Given each pod's memory request, limit, and current usage in MiB, classify it. Usage above the limit means `OOMKilled`. No request means `BestEffort`, which is evicted first. Request equal to limit means `Guaranteed`, which is evicted last. Anything else is `Burstable`, ranked by usage above the request.

**Hint.** Check the limit first, then whether a request exists, then whether request equals limit. For `Burstable`, the excess is `usage - request`; the largest excess is evicted first.

**Expected result.** `etl` is `OOMKilled`, `api` is `Burstable` with `876 MiB` of excess, `cache` is `Guaranteed`, and `scratch` is `BestEffort`. `api` is the first Burstable pod to be evicted.

In [ ]:
def eviction_risk(pod):
    if pod["limit_mib"] and pod["usage_mib"] > pod["limit_mib"]:
        return "OOMKilled", 0
    if not pod["request_mib"]:
        return "BestEffort", 0
    if pod["request_mib"] == pod["limit_mib"]:
        return "Guaranteed", 0
    return "Burstable", pod["usage_mib"] - pod["request_mib"]


pods = [
    {"name": "etl", "request_mib": 2048, "limit_mib": 4096, "usage_mib": 4600},
    {"name": "api", "request_mib": 1024, "limit_mib": 2048, "usage_mib": 1900},
    {"name": "cache", "request_mib": 4096, "limit_mib": 4096, "usage_mib": 3900},
    {"name": "scratch", "request_mib": 0, "limit_mib": 0, "usage_mib": 800},
]

burstable = []
for pod in pods:
    kind, excess = eviction_risk(pod)
    print(f"{pod['name']:<8} {kind:<11} excess over request = {excess} MiB")
    if kind == "Burstable":
        burstable.append((pod["name"], excess))

burstable.sort(key=lambda item: item[1], reverse=True)
print("first Burstable to be evicted:", burstable[0][0])

assert eviction_risk(pods[0]) == ("OOMKilled", 0)
assert eviction_risk(pods[1]) == ("Burstable", 876)
assert eviction_risk(pods[2]) == ("Guaranteed", 0)
assert eviction_risk(pods[3]) == ("BestEffort", 0)
assert burstable[0] == ("api", 876)

### Try it 7. Plan a CronJob overlap policy

**Task.** An hourly CronJob takes 75 minutes at p95. Simulate 24 hours under `Allow`, `Forbid`, and `Replace`, and report starts, skipped runs, and peak concurrency. Then pick the policy for a job that writes the same target partition.

**Hint.** At each scheduled tick, drop finished runs, then apply the policy: `Allow` starts anyway, `Forbid` skips the tick while a run is active, and `Replace` cancels the active run and starts a new one.

**Expected result.** `Allow` starts 24 with peak 2; `Forbid` starts 12 and skips 12 with peak 1; `Replace` starts 24 and cancels the active run each hour, so peak is 1. `Forbid` is the right choice for same-target writes.

In [ ]:
def simulate(policy, hours=24, period_minutes=60, duration_minutes=75):
    running = []
    starts = skips = peak = 0
    for tick in range(0, hours * 60, period_minutes):
        running = [end for end in running if end > tick]
        if running:
            if policy == "Forbid":
                skips += 1
                continue
            if policy == "Replace":
                running = []
        starts += 1
        running.append(tick + duration_minutes)
        peak = max(peak, len(running))
    return starts, skips, peak


results = {policy: simulate(policy) for policy in ("Allow", "Forbid", "Replace")}
for policy, (starts, skips, peak) in results.items():
    print(f"{policy:<8} starts={starts:2d} skips={skips:2d} peak_concurrency={peak}")

assert results["Allow"] == (24, 0, 2)
assert results["Forbid"] == (12, 12, 1)
assert results["Replace"] == (24, 0, 1)
assert results["Forbid"][1] == 12

## Real world use cases for Kubernetes

### 1. Spark batch platform on EKS with spot node pools

A retail data team runs 300-500 Spark jobs a night (2-40 minutes each, 30-80 executors) over a 900 TB lake on S3. Steady-state on-demand capacity would be idle most of the day.

```bash
spark-submit \
  --master k8s://https://<eks-api>:443 \
  --deploy-mode cluster \
  --conf spark.kubernetes.namespace=spark-batch \
  --conf spark.kubernetes.container.image=registry.example.com/data/spark:3.5.3 \
  --conf spark.kubernetes.executor.podTemplateFile=spot-executor.yaml \
  --conf spark.dynamicAllocation.enabled=true \
  --conf spark.dynamicAllocation.maxExecutors=200 \
  s3://acme-code/jobs/nightly_batch.py
```

```yaml
# spot-executor.yaml
apiVersion: v1
kind: Pod
metadata:
  labels: {app: spark-executor}
spec:
  nodeSelector:
    capacity-type: spot
  tolerations:
    - key: capacity-type
      operator: Equal
      value: spot
      effect: NoSchedule
  terminationGracePeriodSeconds: 120
```

Why this design: executors are ephemeral, so spot interruption is tolerable - Spark retries the lost tasks. Cluster autoscaler + Karpenter add nodes for the nightly burst and release them by morning, which is the cost lever. Drivers run in-cluster so CI and laptops can disconnect. The trade-off is task re-execution and shuffle loss on interruption, which is why jobs checkpoint to S3 and use dynamic allocation instead of fixed executor counts.

### 2. Multi-tenant Airflow on Kubernetes with per-team isolation

A fintech central platform team serves 14 data teams on one Airflow deployment. Each task runs in its own pod with the team's image and service account, and each namespace has a quota.

```yaml
apiVersion: v1
kind: ResourceQuota
metadata:
  name: team-payments-quota
  namespace: airflow-payments
spec:
  hard:
    requests.cpu: "40"
    requests.memory: 160Gi
    limits.cpu: "80"
    limits.memory: 320Gi
    pods: "120"
```

```yaml
# the task pod template points at the team's service account
spec:
  serviceAccountName: airflow-payments
  nodeSelector:
    node-pool: general
```

Why this design: KubernetesExecutor gives per-task image and resource isolation, so the payments team can pin a different Spark client or Python version without breaking another team. RBAC plus namespace quotas stop one team's runaway backfill from starving the rest. The trade-off is 10-30 seconds of pod startup per task - fine for distributed stores, painful for thousands of tiny tasks, which is why small tasks stay on a Celery pool.

### 3. Streaming ingestion with Kafka/Flink operators

A logistics company ingests 120k events/second from 40k vehicles through Kafka (retention 7 days) and enriches them with a Flink job. Both clusters are managed by operators rather than hand-written StatefulSets.

```yaml
apiVersion: flink.apache.org/v1beta1
kind: FlinkDeployment
metadata:
  name: vehicle-enrichment
  namespace: streaming
spec:
  image: registry.example.com/data/flink:1.20
  flinkVersion: v1_20
  serviceAccount: flink
  jobManager:
    resource: {cpu: 2, memory: "4g"}
    replicas: 2
  taskManager:
    resource: {cpu: 4, memory: "8g"}
  job:
    jarURI: s3://acme-code/flink/vehicle-enrichment.jar
    parallelism: 24
    upgradeMode: savepoint
    state: running
```

Why this design: the operator handles the ordered rolling upgrade with a savepoint, restarts on failure, and rebalances partitions - the work a plain Deployment cannot do safely. A 2-JM / 24-TM shape tolerates one JM loss and scales with the event rate. The trade-off is that operator CRDs are another upgrade surface and the team must understand savepoint compatibility before changing parallelism or state.

### 4. Self-service dbt runs as Kubernetes Jobs

A 60-person analytics org has 900 dbt models. Instead of a central Airflow DAG, a service accepts a run request and creates a Kubernetes Job with the requested selector.

```yaml
apiVersion: batch/v1
kind: Job
metadata:
  generateName: dbt-run-
  namespace: analytics
spec:
  backoffLimit: 0
  ttlSecondsAfterFinished: 3600
  template:
    spec:
      restartPolicy: Never
      containers:
        - name: dbt
          image: registry.example.com/data/dbt:1.9
          command: ["dbt", "build", "--select", "tag:finance", "--target", "prod"]
          env:
            - name: DBT_PROFILES_DIR
              value: /opt/dbt
            - name: DBT_SNOWFLAKE_PASSWORD
              valueFrom:
                secretKeyRef: {name: dbt-snowflake, key: password}
          resources:
            requests: {cpu: "500m", memory: 1Gi}
            limits: {cpu: "1", memory: 2Gi}
```

Why this design: each run is isolated, logged, and garbage-collected by TTL, and the image pins the dbt version so a model change cannot break another team's run. `backoffLimit: 0` makes a failed run fail fast and surface the error instead of silently retrying a bad model. The trade-off is no built-in DAG-level dependency management; the caller (Airflow, a CI job, or a small control plane) owns ordering.